# Transfer Learning from ResNet50

In [1]:
import torch, torchvision
import numpy as np
from collect_data import collect_data, train_model

DATA_DIR = '../data/images/climbs'

# Load pretrained ResNet model
resnet50 = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)

# Transform images to batch_size x channels x rows x columns
preprocess = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Get training and validation data
train_loader, val_loader, test_loader = collect_data(DATA_DIR, preprocess)

In [2]:
print(resnet50)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [3]:
class ResNetTransfer(torch.nn.Module):
    def __init__(self, resnetmodel):
        super(ResNetTransfer, self).__init__()
        # ResNet structure (without classifier)
        self.resnet = torch.nn.Sequential(resnetmodel.conv1,
                                            resnetmodel.bn1,
                                            resnetmodel.relu,
                                            resnetmodel.maxpool,
                                            resnetmodel.layer1,
                                            resnetmodel.layer2,
                                            resnetmodel.layer3,
                                            resnetmodel.layer4,
                                            resnetmodel.avgpool)
        # 2048 from the resnet convolution and 1 from the angles
        self.classifier = torch.nn.Sequential(torch.nn.Linear(2049, 23))
    
    def forward(self, X, angles):
        r = self.resnet(X)
        f = torch.flatten(r, 1)
        f = torch.cat([f, angles], dim=1)
        y = self.classifier(f)
        return y

In [4]:
device = torch.device('cpu') 
if torch.cuda.is_available():
    device = torch.device('cuda') # rose laptops and the servers
elif torch.backends.mps.is_available():
    device = torch.device('mps') # mac
print('Using device:', device)

Using device: cuda


In [5]:
import copy

model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)

# Freeze the pretrained convolutional feature extractor. 
# Only the new Linear classifier gets trained.
for param in model.resnet.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

Total trainable parameters: 47150


In [6]:
# Hyperparameters
epsilon = 1E-4

# Use CCE loss
criterion = torch.nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=epsilon)

train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=100, checkpoint_path='resnet_transfer_model.pth')

Epoch [0/100], Train Loss: 3.38267, Val Loss: 3.36781 *, Train Acc: 0.00000, Validation Acc: 0.06667
Epoch [1/100], Train Loss: 3.17049, Val Loss: 3.26378 *, Train Acc: 0.01429, Validation Acc: 0.06667
Epoch [2/100], Train Loss: 3.07251, Val Loss: 3.15114 *, Train Acc: 0.05714, Validation Acc: 0.06667
Epoch [3/100], Train Loss: 3.05520, Val Loss: 3.02679 *, Train Acc: 0.08571, Validation Acc: 0.06667
Epoch [4/100], Train Loss: 2.84699, Val Loss: 2.92264 *, Train Acc: 0.14286, Validation Acc: 0.06667
Epoch [5/100], Train Loss: 2.86554, Val Loss: 2.83732 *, Train Acc: 0.20000, Validation Acc: 0.06667
Epoch [6/100], Train Loss: 2.69662, Val Loss: 2.75534 *, Train Acc: 0.18571, Validation Acc: 0.06667
Epoch [7/100], Train Loss: 2.69509, Val Loss: 2.67724 *, Train Acc: 0.21429, Validation Acc: 0.06667
Epoch [8/100], Train Loss: 2.65366, Val Loss: 2.59187 *, Train Acc: 0.24286, Validation Acc: 0.33333
Epoch [9/100], Train Loss: 2.65722, Val Loss: 2.51940 *, Train Acc: 0.27143, Validation Acc

In [7]:
# Test model on test split
from test_model import test_model

test_model(ResNetTransfer(copy.deepcopy(resnet50)).to(device), 'resnet_transfer_model.pth', criterion, test_loader, device)

Loss: 2.4329309463500977
Accuracy: 0.3125


In [8]:
# Fine-tuning
finetune_model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)
finetune_model.load_state_dict(torch.load('resnet_transfer_model.pth', map_location=device))

for param in finetune_model.resnet.parameters():
    param.requires_grad = True

# Use different learning rate on different params to train resnet layers more slowly
finetune_optimizer = torch.optim.Adam([
    {'params': finetune_model.resnet.parameters(), 'lr': 1e-6},
    {'params': finetune_model.classifier.parameters(), 'lr': epsilon},
])

train_model(finetune_model, train_loader, val_loader, criterion, finetune_optimizer, device, num_epochs=10, checkpoint_path='resnet_finetuned.pth')

Epoch [0/10], Train Loss: 2.60796, Val Loss: 2.47236 *, Train Acc: 0.17143, Validation Acc: 0.53333
Epoch [1/10], Train Loss: 2.56197, Val Loss: 2.49599, Train Acc: 0.17143, Validation Acc: 0.53333
Epoch [2/10], Train Loss: 2.26824, Val Loss: 2.52593, Train Acc: 0.30000, Validation Acc: 0.46667
Epoch [3/10], Train Loss: 2.43201, Val Loss: 2.54390, Train Acc: 0.11429, Validation Acc: 0.46667
Epoch [4/10], Train Loss: 2.53460, Val Loss: 2.56077, Train Acc: 0.21429, Validation Acc: 0.46667
Epoch [5/10], Train Loss: 2.28484, Val Loss: 2.58915, Train Acc: 0.24286, Validation Acc: 0.46667
Epoch [6/10], Train Loss: 2.37829, Val Loss: 2.63178, Train Acc: 0.18571, Validation Acc: 0.46667
Epoch [7/10], Train Loss: 2.26007, Val Loss: 2.68375, Train Acc: 0.32857, Validation Acc: 0.46667
Epoch [8/10], Train Loss: 2.42577, Val Loss: 2.73171, Train Acc: 0.30000, Validation Acc: 0.26667
Epoch [9/10], Train Loss: 2.26490, Val Loss: 2.77946, Train Acc: 0.32857, Validation Acc: 0.06667
Average time per e

In [10]:
# Test model on test split
from test_model import test_model

test_model(ResNetTransfer(copy.deepcopy(resnet50)).to(device), 'resnet_transfer_model.pth', criterion, test_loader, device)

Loss: 2.4329309463500977
Accuracy: 0.3125
